# 01 - Strict OpenAlex name matching

This notebook matches PC researchers to OpenAlex Author IDs using one
simple rule: the normalized mapped PC researcher name must appear in the
exploded cited author table, and that normalized name must map to exactly
one OpenAlex Author ID.

My rule is conservative. It gives a high precision match table and
leaves all unmatched or ambiguous names for validation or manual review.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from author_matching import build_pc_people, normalize_name, short_openalex_id
from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_1_PREPARED = PROJECT / "step_1_data" / "prepared"
STEP_2_PREPARED = PROJECT / "step_2_data" / "prepared" / "all_papers"
STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
SUMMARY_TABLES = PROJECT / "step_3_artifacts" / "summary_tables"
CHECK_TABLES = PROJECT / "step_3_artifacts" / "check_tables"

ensure_dirs(STEP_3_PREPARED, SUMMARY_TABLES, CHECK_TABLES)

STEP_1_DEPENDENCIES = PROJECT / "step_1_artifacts" / "dependency_tables"

PC_MEMBERS_PATH = STEP_1_PREPARED / "pc_members.parquet"
NAME_MAP_PATH = STEP_1_DEPENDENCIES / "name_map_used_for_source_comparison.csv"
REF_AUTHORS_PATH = STEP_2_PREPARED / "all_ref_authors_exploded.parquet"

MATCH_OUT = STEP_3_PREPARED / "pc_members_openalex_match.parquet"
MATCH_REVIEW_OUT = CHECK_TABLES / "openalex_strict_name_match_review_cases.csv"
MATCH_DETAIL_OUT = CHECK_TABLES / "openalex_strict_name_match_details.csv"
MATCH_SUMMARY_OUT = SUMMARY_TABLES / "openalex_strict_name_match_summary.csv"

print(f"Project folder: {PROJECT}")
print(f"Run mode: {setup.run_mode}")

Project folder: /Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2. Read the inputs

In [2]:
pc_members = pd.read_parquet(PC_MEMBERS_PATH)
name_map = pd.read_csv(NAME_MAP_PATH)
name_map_dict = dict(zip(name_map["name_variant"], name_map["name_canonical"]))
ref_authors = pd.read_parquet(
    REF_AUTHORS_PATH,
    columns=["ref_author_name", "ref_author_id", "ref_orcid", "work_id", "referenced_work_id"],
)

print(f"PC-service rows: {len(pc_members):,}")
print(f"Unique PC researchers: {pc_members['canonical_researchr_id'].nunique():,}")
print(f"Reference-author rows: {len(ref_authors):,}")

display(pc_members.head())
display(name_map.head())
display(ref_authors.head())

PC-service rows: 2,180
Unique PC researchers: 952
Reference-author rows: 355,366


,service_key,conference,year,researcher_id,name,role,affiliation,country,person_url,researchr_id,source_canonical_researchr_id,canonical_researchr_id,committee_source,source,is_visible_pc,pc_it,source_url,final_url
0,ICFP_2017_adamchlipala,ICFP,2017,adamchlipala,Adam Chlipala,PC Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala,adamchlipala,adamchlipala,adamchlipala,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
1,ICFP_2017_alanjeffrey1,ICFP,2017,alanjeffrey1,Alan Jeffrey,PC Member,Mozilla Research,United States,https://icfp17.sigplan.org/profile/alanjeffrey1,alanjeffrey1,alanjeffrey1,alanjeffrey1,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
2,ICFP_2017_alexandrasilva,ICFP,2017,alexandrasilva,Alexandra Silva,PC Member,University College London,United Kingdom,https://icfp17.sigplan.org/profile/alexandrasilva,alexandrasilva,alexandrasilva,alexandrasilva,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
3,ICFP_2017_benlippmeier,ICFP,2017,benlippmeier,Ben Lippmeier,PC Member,Digital Asset / UNSW Australia,,https://icfp17.sigplan.org/profile/benlippmeier,benlippmeier,benlippmeier,benlippmeier,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
4,ICFP_2017_betaziliani,ICFP,2017,betaziliani,Beta Ziliani,PC Member,"FAMAF, UNC and CONICET",Argentina,https://icfp17.sigplan.org/profile/betaziliani,betaziliani,betaziliani,betaziliani,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...


,name_variant,name_canonical
0,Aditya Thakur,Aditya V. Thakur
1,Alastair Donaldson,Alastair F. Donaldson
2,Alexander Lew,Alexander K. Lew
3,Amir Kafshdar Goharshady,Amir K. Goharshady
4,Andrew Pitts,Andrew M. Pitts


,ref_author_name,ref_author_id,ref_orcid,work_id,referenced_work_id
0,Przemysław Prusinkiewicz,A5086990150,0000-0002-1338-7086,W2753491390,W1539916920
1,Aristid Lindenmayer,A5052731881,NaN,W2753491390,W1564690864
2,John Corbit,A5079085452,0000-0002-9099-6272,W2753491390,W1966244296
3,David J. Garbary,A5062636945,0000-0001-5126-6608,W2753491390,W1966244296
4,Przemysław Prusinkiewicz,A5086990150,0000-0002-1338-7086,W2753491390,W2000690667


## 3. Build one row per PC researcher

The PC service file has one row per service record. For author matching,
I collapse it to one row per `canonical_researchr_id` and apply the
small project name map before exact matching.

In [3]:
pc_people = build_pc_people(pc_members)
pc_people["mapped_name_for_matching"] = (
    pc_people["name"].map(name_map_dict).fillna(pc_people["name"])
)
pc_people["pc_name_norm"] = pc_people["mapped_name_for_matching"].map(normalize_name)

print(f"Researchers to match: {len(pc_people):,}")
display(pc_people.head())

Researchers to match: 952


,canonical_researchr_id,name,name_variants,normalized_names,n_pc_rows,n_conferences,first_observed_pc_year,last_observed_pc_year,mapped_name_for_matching,pc_name_norm
0,aaronbembenek,Aaron Bembenek,[Aaron Bembenek],[aaron bembenek],1,1,2025,2025,Aaron Bembenek,aaron bembenek
1,aaronstump,Aaron Stump,[Aaron Stump],[aaron stump],2,1,2019,2023,Aaron Stump,aaron stump
2,abhinavverma1,Abhinav Verma,[Abhinav Verma],[abhinav verma],1,1,2023,2023,Abhinav Verma,abhinav verma
3,adamchlipala,Adam Chlipala,[Adam Chlipala],[adam chlipala],8,3,2017,2025,Adam Chlipala,adam chlipala
4,adamwelc,Adam Welc,[Adam Welc],[adam welc],2,1,2020,2021,Adam Welc,adam welc


## 4. Build the cited author name index

I group the exploded cited author table by normalized cited author name.
A PC researcher is accepted only when the matching cited author name has
exactly one OpenAlex Author ID.

In [4]:
def unique_list(values, limit=None):
    clean = sorted({
        str(value)
        for value in values
        if pd.notna(value) and str(value).strip()
    })
    if limit is None:
        return clean
    return clean[:limit]


ref = ref_authors.copy()
ref["ref_author_id"] = ref["ref_author_id"].map(short_openalex_id)
ref["ref_orcid"] = ref["ref_orcid"].map(lambda x: x.rstrip("/").split("/")[-1] if isinstance(x, str) and x else None)
ref["ref_norm"] = ref["ref_author_name"].map(normalize_name)
ref = ref[(ref["ref_norm"] != "") & ref["ref_author_id"].notna()].copy()

cited_name_index = (
    ref.groupby("ref_norm", dropna=False)
    .agg(
        n_openalex_ids=("ref_author_id", "nunique"),
        n_evidence_rows=("ref_author_id", "size"),
        n_citing_papers=("work_id", "nunique"),
        n_cited_works=("referenced_work_id", "nunique"),
        candidate_openalex_ids=("ref_author_id", lambda x: unique_list(x)),
        candidate_ref_author_names=("ref_author_name", lambda x: unique_list(x, limit=10)),
        candidate_orcids=("ref_orcid", lambda x: unique_list(x)),
    )
    .reset_index()
)

print(f"Normalized cited-author names: {len(cited_name_index):,}")
display(cited_name_index.head())

Normalized cited-author names: 48,230


,ref_norm,n_openalex_ids,n_evidence_rows,n_citing_papers,n_cited_works,candidate_openalex_ids,candidate_ref_author_names,candidate_orcids
0,:,1,3,2,3,[A9999999999],[:],[]
1,a a gaffar,1,1,1,1,[A5110268107],[A.A. Gaffar],[]
2,a a orlikovsky,1,1,1,1,[A5113951453],[A. A. Orlikovsky],[]
3,a a terekhov,1,3,3,1,[A5102166885],[A.A. Terekhov],[]
4,a agung julius,1,1,1,1,[A5029921181],[A. Agung Julius],[0000-0002-0970-3226]


## 5. Apply the strict one name / one ID rule

In [5]:
matches = pc_people.merge(
    cited_name_index,
    left_on="pc_name_norm",
    right_on="ref_norm",
    how="left",
)

matches["n_openalex_ids"] = matches["n_openalex_ids"].fillna(0).astype(int)
for column in ["n_evidence_rows", "n_citing_papers", "n_cited_works"]:
    matches[column] = matches[column].fillna(0).astype(int)

def first_or_none(values):
    if isinstance(values, list) and len(values) == 1:
        return values[0]
    return None

matches["openalex_id"] = matches["candidate_openalex_ids"].map(first_or_none)
matches["orcid"] = matches["candidate_orcids"].map(first_or_none)
matches["openalex_matched"] = matches["n_openalex_ids"] == 1
matches.loc[~matches["openalex_matched"], ["openalex_id", "orcid"]] = None
matches["match_source"] = matches["openalex_matched"].map(
    {True: "strict_name_one_openalex_id", False: "unmatched_or_ambiguous_name"}
)
matches["match_confidence"] = matches["openalex_matched"].map(
    {True: "high", False: "review"}
)
matches["needs_manual_review"] = ~matches["openalex_matched"]

def review_reason(row):
    if row["n_openalex_ids"] == 0:
        return "PC name not found with an OpenAlex Author ID in the cited-author table."
    return "PC name maps to multiple OpenAlex Author IDs in the cited-author table."

matches["review_reason"] = matches.apply(
    lambda row: None if row["openalex_matched"] else review_reason(row),
    axis=1,
)

keep_columns = [
    "canonical_researchr_id",
    "name",
    "mapped_name_for_matching",
    "pc_name_norm",
    "openalex_id",
    "orcid",
    "openalex_matched",
    "match_source",
    "match_confidence",
    "needs_manual_review",
    "review_reason",
    "n_evidence_rows",
    "n_citing_papers",
    "n_cited_works",
    "n_openalex_ids",
    "candidate_openalex_ids",
    "candidate_ref_author_names",
    "candidate_orcids",
    "n_pc_rows",
    "n_conferences",
    "first_observed_pc_year",
    "last_observed_pc_year",
]

strict_matches = matches[keep_columns].sort_values(
    ["openalex_matched", "name"],
    ascending=[False, True],
)

print(
    f"Strict matches: {strict_matches['openalex_matched'].sum():,} / {len(strict_matches):,}"
)
display(strict_matches.head())
display(
    strict_matches["review_reason"]
    .fillna("strict one-name / one-ID match")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="n_researchers")
)

Strict matches: 793 / 952


,canonical_researchr_id,name,mapped_name_for_matching,pc_name_norm,openalex_id,orcid,openalex_matched,match_source,match_confidence,needs_manual_review,...,n_citing_papers,n_cited_works,n_openalex_ids,candidate_openalex_ids,candidate_ref_author_names,candidate_orcids,n_pc_rows,n_conferences,first_observed_pc_year,last_observed_pc_year
1,aaronstump,Aaron Stump,Aaron Stump,aaron stump,A5072489480,0000-0002-9720-0003,True,strict_name_one_openalex_id,high,False,...,44,21,1,[A5072489480],[Aaron Stump],[0000-0002-9720-0003],2,1,2019,2023
2,abhinavverma1,Abhinav Verma,Abhinav Verma,abhinav verma,A5101988843,0000-0002-9820-8285,True,strict_name_one_openalex_id,high,False,...,1,1,1,[A5101988843],[Abhinav Verma],[0000-0002-9820-8285],1,1,2023,2023
3,adamchlipala,Adam Chlipala,Adam Chlipala,adam chlipala,A5078100439,0000-0001-7085-9417,True,strict_name_one_openalex_id,high,False,...,235,61,1,[A5078100439],[Adam Chlipala],[0000-0001-7085-9417],8,3,2017,2025
4,adamwelc,Adam Welc,Adam Welc,adam welc,A5078668785,0009-0005-0515-4994,True,strict_name_one_openalex_id,high,False,...,23,11,1,[A5078668785],[Adam Welc],[0009-0005-0515-4994],2,1,2020,2021
7,adriansampson,Adrian Sampson,Adrian Sampson,adrian sampson,A5004782337,0000-0003-0837-8924,True,strict_name_one_openalex_id,high,False,...,43,23,1,[A5004782337],[Adrian Sampson],[0000-0003-0837-8924],3,2,2018,2020


,status,n_researchers
0,strict one-name / one-ID match,793
1,PC name maps to multiple OpenAlex Author IDs i...,84
2,PC name not found with an OpenAlex Author ID i...,75


## 6. Save outputs

In [6]:
review_cases = strict_matches[strict_matches["needs_manual_review"]].copy()

summary = (
    strict_matches["review_reason"]
    .fillna("strict one-name / one-ID match")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="n_researchers")
)
summary["share_of_pc_researchers"] = summary["n_researchers"] / len(strict_matches)

def write_parquet(path, frame):
    if path.exists() and not setup.overwrite_data:
        print(f"Keeping existing data file: {path.relative_to(PROJECT)}")
        return
    frame.to_parquet(path, index=False)
    print(f"Wrote {path.relative_to(PROJECT)}: {frame.shape}")


def write_csv(path, frame):
    if path.exists() and not setup.overwrite_artifacts:
        print(f"Keeping existing artifact: {path.relative_to(PROJECT)}")
        return
    frame.to_csv(path, index=False)
    print(f"Wrote {path.relative_to(PROJECT)}: {frame.shape}")


write_parquet(MATCH_OUT, strict_matches)
write_csv(MATCH_DETAIL_OUT, strict_matches)
write_csv(MATCH_REVIEW_OUT, review_cases)
write_csv(MATCH_SUMMARY_OUT, summary)

Keeping existing data file: step_3_data/prepared/pc_members_openalex_match.parquet
Wrote step_3_artifacts/check_tables/openalex_strict_name_match_details.csv: (952, 22)
Wrote step_3_artifacts/check_tables/openalex_strict_name_match_review_cases.csv: (159, 22)
Wrote step_3_artifacts/summary_tables/openalex_strict_name_match_summary.csv: (3, 3)


## 7. My Interpretation

This strict match is easy to check: a researcher is matched only when the
mapped PC name corresponds to exactly one OpenAlex Author ID in the
exploded cited author data. ORCID and DOI checks can validate these
matches later, but they do not add extra matches in this notebook.